<a name="top" id="top"></a>

<div align="center">
    <h1>Deterministic optimization</h1>
    <a href="https://github.com/sa1K">Sai Karthik</a>
    <br>
    <i>Weldon School of Biomedical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://github.com/parkyr">Yirang Park</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://github.com/bernalde">David E. Bernal Neira</a>
    <br>
    <i>Davidson School of Chemical Engineering, Purdue University</i>
    <br>
    <br>
    <a href="https://colab.research.google.com/github/SECQUOIA/Pharma-optimization-flowsheet/blob/main/Enhanced_model/Pharma_Optimizer.ipynb" target="_parent">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
    <a href="https://secquoia.github.io/">
        <img src="https://img.shields.io/badge/🌲⚛️🌐-SECQUOIA-blue" alt="SECQUOIA"/>
    </a>
</div>

# Introduction

This notebook formulates the multi-echelon pharmaceutical vendor-selection and transportation-routing problem as a **mixed-integer linear program (MILP)** and solves it to find the minimum-cost supply-chain configuration for a fixed, known demand. Given a set of manufacturing steps (e.g. API synthesis → formulation → fill/finish), a set of candidate vendors at each step with their per-batch costs and capacities, and a transportation-cost matrix between vendors at consecutive steps, the model decides:

- **Which vendor to contract at each step** (a binary selection variable per vendor).
- **Which transportation routes to use** between the selected vendors (a binary selection per route).
- **How much material flows through each step** (continuous throughput variables, bounded by the selected vendor's capacity).

These choices are coupled by the usual network-flow constraints — exactly one vendor per step, routes only active between selected vendors, flow non-increasing along the chain, and final-step throughput ≥ demand — and the objective minimizes the sum of production and transportation costs.

### Why an optimization solver rather than enumeration?

The combinatorial structure of the problem (pick one vendor per step, then pick one route per consecutive pair) makes exhaustive enumeration impractical at realistic sizes: with $N$ steps and $V$ vendors per step the vendor combinations alone scale as $V^N$, and each combination carries its own route-cost selection. A MILP solver like **GLPK** or **HiGHS** navigates this space efficiently using branch-and-bound with LP relaxations, returning a certifiably optimal solution rather than a heuristic one.

### What this notebook covers

1. **Model construction** using [Pyomo](https://pyomo.readthedocs.io/) — sets, parameters, variables, constraints, and the cost objective.
2. **Solving** the MILP with both GLPK and HiGHS and extracting the optimal vendor/route selection.
3. **Visualization** of the chosen path through the supply-chain graph.
4. **Empirical scaling analysis** — how solve time grows as the number of steps and/or vendors per step increases, and how the two solvers compare.

This deterministic formulation serves as the baseline for the companion notebook `Stochastic_Optimization.ipynb`, which extends it to the case where demand is uncertain and must be represented by a set of probability-weighted scenarios.

## Mathematical Formulation

### Sets

| Symbol | Definition |
|--------|------------|
| $\mathcal{S}$ | Set of manufacturing steps (echelons), indexed by $s$ (e.g., API synthesis, formulation, fill/finish) |
| $\mathcal{V}_s$ | Set of candidate vendors at step $s$, indexed by $v$ |
| $\mathcal{R}$ | Set of transport routes between consecutive steps; each route is a tuple $(s_1, v_1, s_2, v_2)$ connecting vendor $v_1$ at step $s_1$ to vendor $v_2$ at step $s_2$ |

### Parameters

| Symbol | Definition | Units |
|--------|------------|-------|
| $c_{s,v}$ | Production cost of vendor $v$ at step $s$ | \$/batch |
| $q_{s,v}$ | Production capacity of vendor $v$ at step $s$ | units |
| $t_{s_1,v_1,s_2,v_2}$ | Transportation cost for route $(s_1,v_1,s_2,v_2) \in \mathcal{R}$ | \$ |
| $d$ | Fixed (known) demand at the final step | units |

### Decision Variables

| Variable | Domain | Definition |
|----------|--------|------------|
| $x_{s,v}$ | Binary $\{0,1\}$ | $= 1$ if vendor $v$ is selected at step $s$, 0 otherwise |
| $y_{s_1,v_1,s_2,v_2}$ | Binary $\{0,1\}$ | $= 1$ if transport route $(s_1,v_1,s_2,v_2)$ is selected, 0 otherwise |
| $f_s$ | Continuous $\geq 0$ | Material flow (throughput) at step $s$ |

### Objective

Minimize the total production and transportation cost:

$$\min \; \sum_{s \in \mathcal{S}} \sum_{v \in \mathcal{V}_s} c_{s,v} \, x_{s,v} \;+\; \sum_{(s_1,v_1,s_2,v_2) \in \mathcal{R}} t_{s_1,v_1,s_2,v_2} \, y_{s_1,v_1,s_2,v_2}$$

### Constraints

1. **One vendor per step:** Exactly one vendor must be selected at each manufacturing step.
$$\sum_{v \in \mathcal{V}_s} x_{s,v} = 1 \qquad \forall \, s \in \mathcal{S}$$

2. **One transport route per step pair:** Exactly one route connects each pair of consecutive steps.
$$\sum_{\substack{(s_1,v_1,s_2,v_2) \in \mathcal{R} \\ s_1 = s,\; s_2 = s'}} y_{s_1,v_1,s_2,v_2} = 1 \qquad \forall \, (s, s') \text{ consecutive in } \mathcal{S}$$

3. **Route–vendor linking:** A transport route can only be selected if both its source and destination vendors are selected.
$$y_{s_1,v_1,s_2,v_2} \leq x_{s_1,v_1} \qquad \forall \, (s_1,v_1,s_2,v_2) \in \mathcal{R}$$
$$y_{s_1,v_1,s_2,v_2} \leq x_{s_2,v_2} \qquad \forall \, (s_1,v_1,s_2,v_2) \in \mathcal{R}$$

4. **Step capacity:** Flow at each step cannot exceed the capacity of the selected vendor.
$$f_s \leq \sum_{v \in \mathcal{V}_s} q_{s,v} \, x_{s,v} \qquad \forall \, s \in \mathcal{S}$$

5. **Flow conservation:** Material flow is non-increasing across consecutive steps (serial supply chain — no inventory buildup between echelons).
$$f_{s'} \leq f_{s} \qquad \forall \, (s, s') \text{ consecutive in } \mathcal{S}$$

6. **Demand satisfaction:** Throughput at the final step $s_{\text{last}}$ must meet demand.
$$f_{s_{\text{last}}} \geq d$$

# Setup and Imports

In [ ]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Install dependencies if in Colab
if IN_COLAB:
    !pip install -q pyomo
    !apt-get install -y -qq glpk-utils
    !pip install highspy
    !sudo apt-get install graphviz graphviz-dev
    !pip install networkx

In [ ]:
import pyomo.environ as pyo
import matplotlib.pyplot as plt
import pandas as pd
import time

# Import from our package
from pharma_optimizer import (
    Generator,
    EnhancedProductionOptimizer,
    visualize_solution_from_optimizer
)

# Data Generation

Generate random manufacturing and transportation data for the optimization problem.

In [ ]:
# Generate a 3-step, 3-vendor instance
generator = Generator(steps=3, options=3)
generator.createManufacturingData()
generator.createTransportData()

In [ ]:
# Load and display production data
prod_df = pd.read_csv("manufacturingData2.csv")
print("Production Data:")
print(prod_df)
print("\n" + "="*50)

# Load and display transport data
transport_df = pd.read_csv("transportData2.csv")
print("\nTransport Data:")
print(transport_df)

# Run Optimization

Solve the deterministic optimization problem to find the minimum cost vendor selection and transportation routing.

In [ ]:
# Create and solve the optimization model
optimizer = EnhancedProductionOptimizer(
    "manufacturingData2.csv",
    "transportData2.csv",
    demand=100
)

# Solve using GLPK solver
if IN_COLAB:
    opt = pyo.SolverFactory('glpk', executable='/usr/bin/glpsol')
else:
    opt = pyo.SolverFactory('glpk')

result = opt.solve(optimizer.model, tee=False)
print(f"Solver status: {result.solver.termination_condition}")

# Display results
optimizer.summary()

# Visualization

Visualize the optimal solution path through the manufacturing network.

In [ ]:
# Generate and display the visualization
G, pos, chosen_path = visualize_solution_from_optimizer(optimizer)
plt.show()

print(f"\nVisualization complete!")
print(f"Found {len(optimizer.vendors)} steps with {sum(len(v) for v in optimizer.vendors.values())} total vendor options")
print(f"Selected path has {len(chosen_path)} steps")

# Solver Scaling Analysis

Compare solver performance across different problem sizes.

In [ ]:
# Scaling: vary both steps and vendors
time_to_run_glpk = []
time_to_run_highs = []

for i in range(5, 35, 5):
    generator = Generator(i, i)
    generator.createManufacturingData()
    generator.createTransportData()

    optimizer = EnhancedProductionOptimizer(
        "manufacturingData2.csv",
        "transportData2.csv",
        demand=100
    )

    # GLPK
    opt = pyo.SolverFactory('glpk')
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_glpk.append(result.solver.time)

    # HiGHS
    opt = pyo.SolverFactory('highs')
    t0 = time.perf_counter()
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_highs.append(time.perf_counter() - t0)

    print(f"Size {i}: GLPK={time_to_run_glpk[-1]:.3f}s, HiGHS={time_to_run_highs[-1]:.3f}s")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(5, 35, 5), time_to_run_glpk, label="GLPK")
ax.scatter(range(5, 35, 5), time_to_run_highs, label="HiGHS")
ax.set_xlabel("Number of vendors and steps", fontsize=14)
ax.set_ylabel("Solution time (seconds)", fontsize=14)
ax.set_title("Solution time for varying both number of vendors and steps", fontsize=14)
ax.legend(fontsize=12)
plt.show()

In [ ]:
# Scaling: vary only steps (fixed 5 vendors)
time_to_run_glpk = []
time_to_run_highs = []

for i in range(5, 35, 5):
    generator = Generator(i, 5)
    generator.createManufacturingData()
    generator.createTransportData()

    optimizer = EnhancedProductionOptimizer(
        "manufacturingData2.csv",
        "transportData2.csv",
        demand=100
    )

    opt = pyo.SolverFactory('glpk')
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_glpk.append(result.solver.time)

    opt = pyo.SolverFactory('highs')
    t0 = time.perf_counter()
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_highs.append(time.perf_counter() - t0)

    print(f"Steps {i}: GLPK={time_to_run_glpk[-1]:.3f}s, HiGHS={time_to_run_highs[-1]:.3f}s")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(5, 35, 5), time_to_run_glpk, label="GLPK")
ax.scatter(range(5, 35, 5), time_to_run_highs, label="HiGHS")
ax.set_xlabel("Number of steps", fontsize=14)
ax.set_ylabel("Solution time (seconds)", fontsize=14)
ax.set_title("Solution time for varying number of steps (5 vendors)", fontsize=14)
ax.legend(fontsize=12)
plt.show()

In [ ]:
# Scaling: vary only vendors (fixed 5 steps)
time_to_run_glpk = []
time_to_run_highs = []

for i in range(5, 35, 5):
    generator = Generator(5, i)
    generator.createManufacturingData()
    generator.createTransportData()

    optimizer = EnhancedProductionOptimizer(
        "manufacturingData2.csv",
        "transportData2.csv",
        demand=100
    )

    opt = pyo.SolverFactory('glpk')
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_glpk.append(result.solver.time)

    opt = pyo.SolverFactory('highs')
    t0 = time.perf_counter()
    result = opt.solve(optimizer.model, tee=False)
    time_to_run_highs.append(time.perf_counter() - t0)

    print(f"Vendors {i}: GLPK={time_to_run_glpk[-1]:.3f}s, HiGHS={time_to_run_highs[-1]:.3f}s")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(range(5, 35, 5), time_to_run_glpk, label="GLPK")
ax.scatter(range(5, 35, 5), time_to_run_highs, label="HiGHS")
ax.set_xlabel("Number of vendors per step", fontsize=14)
ax.set_ylabel("Solution time (seconds)", fontsize=14)
ax.set_title("Solution time for varying number of vendors (5 steps)", fontsize=14)
ax.legend(fontsize=12)
plt.show()